
# Генерация тестовых данных 

Для выполнения ТЗ на позицию Data-analyst предлагается использовать следующую структуру данных:

*	Таблица "users" с полями: id, name, email, created_at

*	Таблица "orders" с полями: id, user_id, total_price, created_at

*	Таблица "order_items" с полями: id, order_id, product_name, price, quantity


In [1]:
import pandas as pd
import numpy as np
import random
from faker import Faker
import datetime as dt

In [2]:
# Используя метод Faker() из библиотеки faker сгенерируем тестовые данные.
fake = Faker() 

In [3]:
n = 1000000 # объявим количество генерируемых значений  

In [4]:
# Создадим пустой список 'names' и с помощью цикла заполним его генерируемыми значениями.
names=[]
for i in range(n):
    names.append(fake.name())

In [5]:
# Создадим пустой список 'emails' и с помощью цикла заполним его генерируемыми значениями.
emails=[]
for i in range(n):
    emails.append(fake.email())

In [6]:
# Создадим пустой список 'dates', зададим период времени, в котором будут генерироваться даты,
# с помощью цикла заполним его генерируемыми значениями.
dates=[]
date_start = '2022-01-01'
date_start = dt.datetime.strptime(date_start, '%Y-%m-%d')
date_end = '2023-04-10'
date_end = dt.datetime.strptime(date_end, '%Y-%m-%d')

for i in range(n):
    dates.append(fake.date_between(date_start, date_end))

In [7]:
# Cоздадим пустой DataFrame, куда и будем добавлять сгенерировынные данные. 
df_users = pd.DataFrame()

In [8]:
# Соберём DataFrame
df_users['name'] = names
df_users['email'] = emails
df_users['created_at'] = dates

In [9]:
# Отсортируем данные по колонке 'created_at'
df_users = df_users.sort_values('created_at', ascending=True)

In [10]:
# Сбросим индекс с удалением
df_users = df_users.reset_index(drop=True)

In [11]:
# Опять сбросим индекс и выведем его в отдельную колонку.
# Переименуем колонку 'index' в 'id'.
df_users = df_users.reset_index()
df_users = df_users.rename(columns={'index': 'id'})

In [12]:
# Так как мы генерируем наши данные для Postgre SQL, учитывая ограничение первичного и внешнего ключей, а также
# разницу в исчислении в SQL и Python, сдвинем все значения столбца "id" на +1.
df_users['id'] = df_users['id'] + 1

In [13]:
df_users.head(2)

,id,name,email,created_at
0,1,Ebony Peters,krobinson@example.net,2022-01-01
1,2,Andrew Davis,robertrivera@example.com,2022-01-01


In [14]:
# Размер df
df_users.shape

(1000000, 4)

In [15]:
# Типы данных df
df_users.dtypes

id             int64
name          object
email         object
created_at    object
dtype: object

In [16]:
# Сохраним DataFrame в файл.csv 
df_users.to_csv('users.csv', index=False)

In [17]:
# Считаем сохранённый файл
df_users = pd.read_csv('users.csv', parse_dates=['created_at'])
df_users.head(2)

,id,name,email,created_at
0,1,Ebony Peters,krobinson@example.net,2022-01-01
1,2,Andrew Davis,robertrivera@example.com,2022-01-01


________________________________________________________________________________________

Таблица "orders" с полями: id, user_id, total_price, created_at

In [18]:
# Таблицу 'orders' сформируем следующим образом:
# 1. ИЗ df_users случайным образом отберём 5 000 000 строк ['id','created_at']
# 2. К каждой дате добавим случайное количество дней из диапазона от 0 до 45
# 3. Общую стоимость сгенерируем случайным образом

In [19]:
df_orders = df_users[['id','created_at']].sample(n * 5, replace = True)

In [20]:
df_orders = df_orders.rename(columns={'id': 'user_id'})

In [21]:
df_orders.shape

(5000000, 2)

In [22]:
df_orders.head(2)

,user_id,created_at
42047,42048,2022-01-20
533905,533906,2022-09-05


In [23]:
df_orders['created_at'] = df_orders.created_at + pd.Timedelta(days = (random.randint(0,45)))

In [24]:
# Создадим список 'prices' и заполним его псевдослучайными целыми числами от 0 до 1000 000 с шагом 10
prices=[]
for i in range(n * 5):
    prices.append(random.randrange(0, 1000000, 10))

In [25]:
# Соберём DataFrame
df_orders['total_price'] = prices
df_orders.head(2)

,user_id,created_at,total_price
42047,42048,2022-02-28,458280
533905,533906,2022-10-14,739570


In [26]:
df_orders = df_orders.sort_values('created_at', ascending=True).reset_index(drop=True)

In [27]:
df_orders = df_orders.reset_index()
df_orders = df_orders.rename(columns={'index': 'id'})

In [28]:
# Так как мы генерируем наши данные для Postgre SQL, учитывая ограничение первичного и внешнего ключей, а также
# разницу в исчислении в SQL и Python, сдвинем все значения столбца "id" на +1.
df_orders['id'] = df_orders['id'] + 1

In [29]:
df_orders.head(2)

,id,user_id,created_at,total_price
0,1,1024,2022-02-09,619280
1,2,539,2022-02-09,143450


In [30]:
# Сохраним DataFrame в файл.csv 
df_orders.to_csv('orders.csv', index=False)

In [31]:
df_orders = pd.read_csv('orders.csv', parse_dates=['created_at'])
df_orders.head(2)

,id,user_id,created_at,total_price
0,1,1024,2022-02-09,619280
1,2,539,2022-02-09,143450


____________________________________________________________________________________________________

По аналогии поступим и с последней таблицей.

Таблица "order_items" с полями: id, order_id, product_name, price, quantity

In [32]:
df_order_items = df_orders[['id','total_price']].sample(n)

In [33]:
df_order_items = df_order_items.rename(columns={'id': 'order_id'})
df_order_items.head()

,order_id,total_price
2673626,2673627,695120
2771899,2771900,677010
1653830,1653831,561110
217248,217249,682610
1802477,1802478,996880


In [34]:
product_name=[]
for i in range(n):
    product_name.append(fake.bothify(text='Prod.№: ????-########'))    

In [35]:
df_order_items['product_name'] = product_name

In [36]:
quantity=[]
for i in range(n):
    quantity.append(random.randrange(1, 30, 1))

In [37]:
df_order_items['quantity'] = quantity

In [38]:
df_order_items['price'] = (df_order_items.total_price / quantity).round(2)

In [39]:
df_order_items = df_order_items[['order_id', 'product_name', 'price', 'quantity']]
df_order_items.head(2)

,order_id,product_name,price,quantity
2673626,2673627,Prod.№: NPuG-25470804,695120.00,1
2771899,2771900,Prod.№: QjBH-91429237,52077.69,13


In [40]:
df_order_items = df_order_items.reset_index(drop=True)

In [41]:
df_order_items = df_order_items.reset_index()
df_order_items = df_order_items.rename(columns={'index': 'id'})


In [42]:
# Так как мы генерируем наши данные для Postgre SQL, учитывая ограничение первичного и внешнего ключей, а также
# разницу в исчислении в SQL и Python, сдвинем все значения столбца "id" на +1.
df_order_items['id'] = df_order_items['id'] + 1

In [43]:
df_order_items.head(2)

,id,order_id,product_name,price,quantity
0,1,2673627,Prod.№: NPuG-25470804,695120.00,1
1,2,2771900,Prod.№: QjBH-91429237,52077.69,13


In [44]:
# Сохраним DataFrame в файл.csv 
df_order_items.to_csv('order_items.csv', index=False)

In [45]:
df_order_items = pd.read_csv('order_items.csv')
df_order_items.head(2)

,id,order_id,product_name,price,quantity
0,1,2673627,Prod.№: NPuG-25470804,695120.00,1
1,2,2771900,Prod.№: QjBH-91429237,52077.69,13
